# Mini Project: Drug Type Prediction using Machine Learning — FIXED VERSION

This notebook follows the professor's original Tasks 1–11 in the same order, while fixing the technical issues in the existing `Progress_of_my_work.docx`.

## Important corrections applied
- Use an **80/20 stratified train/test split** consistently.
- Keep the original `Drug` labels as strings instead of manually mapping LabelEncoder integers.
- Do **not fit preprocessing on the full dataset before splitting**.
- Use **SMOTENC** because the predictors contain numerical and categorical variables.
- Generate **1,000 training samples** from the 160-row training set, which creates 840 synthetic samples and satisfies the ≥800 requirement.
- Use a single sklearn/imbalanced-learn prediction pipeline so the Streamlit application receives the same raw inputs as training.
- Use `model.classes_` for probability labels instead of hard-coded class ordering.
- Build one master model-comparison table instead of duplicating models/results.
- Tune the final candidates without claiming an improvement when the measured result is unchanged or worse.
- Save the complete preprocessing + model pipeline.

# Mini Project: Drug Type Prediction using Machine Learning — PROFESSOR-ALIGNED FIXED VERSION

## How this notebook is organized
This notebook follows the professor's original Task 1–11 structure and keeps the project terminology. Each task contains a short description/note explaining why the code is being performed.

### Corrections applied to the existing project
- Original dataset remains 200 records.
- Train/test split is consistently 80/20: 160 training + 40 testing.
- Target `Drug` is label encoded; predictor categories are one-hot encoded inside the preprocessing pipeline.
- Numerical transformation/scaling is kept inside the modeling pipeline instead of being fitted on the full dataset before the split.
- SMOTENC is used for mixed numerical/categorical augmentation.
- Augmented training data is exactly 1,000 rows: 160 original training + 840 synthetic. The 40-row test set is untouched.
- All required classifiers from the professor's list are evaluated.
- Required evaluation metrics and overfitting checks are calculated from the actual run.
- Hyperparameter tuning results are calculated dynamically.
- The final comparison table contains each model once.
- Feature importance and permutation importance are included; SHAP is optional.
- Streamlit loads the complete preprocessing + model package so prediction uses exactly the training-time preprocessing.

### Important
Do not copy old numeric results such as the previous 98.33% or 96.67% scores into the final report. Re-run this corrected notebook from Task 1 to Task 11 and use the newly calculated outputs.


## Task 1 — Import and Load the Data

### Professor requirements
Import pandas, numpy, matplotlib, seaborn and sklearn; load the dataset; use `.head()`, `.info()`, `.describe()`, `.shape`, and `.columns`.


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Google Colab / Google Drive path used by your existing project.
DATA_PATH = Path("/content/drive/MyDrive/ML Project/drug200.csv")

# If the file is uploaded locally instead, this fallback can be used.
if not DATA_PATH.exists():
    DATA_PATH = Path("drug200.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "drug200.csv was not found. Put it in /content/drive/MyDrive/ML Project/ "
        "or upload it to the notebook runtime."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nData information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all"))

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

## Task 2 — Exploratory Data Analysis (EDA)

### Professor requirements
Use histograms, KDE plots, boxplots, categorical count plots, target distribution, feature-vs-drug relationships, numerical correlation heatmap, and a written EDA summary.


In [ ]:
# Numerical distributions: Age and Na_to_K
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Age"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of Patient Age")

sns.histplot(df["Na_to_K"], kde=True, ax=axes[1])
axes[1].set_title("Distribution of Na_to_K Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# KDE plot for Na_to_K
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="Na_to_K", fill=True)
plt.title("KDE Plot: Na_to_K Ratio")
plt.show()

In [ ]:
# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df["Age"], ax=axes[0])
axes[0].set_title("Boxplot of Age")

sns.boxplot(x=df["Na_to_K"], ax=axes[1])
axes[1].set_title("Boxplot of Na_to_K Ratio")

plt.tight_layout()
plt.show()

In [ ]:
# Categorical count plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.countplot(data=df, x="Sex", ax=axes[0])
axes[0].set_title("Gender Distribution")

sns.countplot(data=df, x="BP", ax=axes[1])
axes[1].set_title("Blood Pressure Distribution")

sns.countplot(data=df, x="Cholesterol", ax=axes[2])
axes[2].set_title("Cholesterol Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Target distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Drug", order=df["Drug"].value_counts().index)
plt.title("Distribution of Target Variable: Drug Type")
plt.xlabel("Drug Type")
plt.ylabel("Number of Patients")
plt.show()

print("Target class counts:")
display(df["Drug"].value_counts().rename("Count").to_frame())

In [ ]:
# Relationships between patient characteristics and Drug

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

sns.swarmplot(data=df, x="Drug", y="Na_to_K", ax=axes[0, 0], size=5)
axes[0, 0].set_title("Na_to_K vs Drug")

sns.boxplot(data=df, x="Drug", y="Age", ax=axes[0, 1])
axes[0, 1].set_title("Age vs Drug")

sns.countplot(data=df, x="BP", hue="Drug", ax=axes[1, 0])
axes[1, 0].set_title("Blood Pressure vs Drug")

sns.countplot(data=df, x="Cholesterol", hue="Drug", ax=axes[1, 1])
axes[1, 1].set_title("Cholesterol vs Drug")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Sex", hue="Drug")
plt.title("Sex vs Drug")
plt.show()

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(6, 4))
sns.heatmap(
    df[["Age", "Na_to_K"]].corr(),
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)
plt.title("Correlation Heatmap: Numerical Features")
plt.show()

### EDA summary — fill this with the actual output from your run

Record the important findings after running the cells:

1. Original dataset size: **200 records**.
2. Identify whether the Drug target is imbalanced and show the exact class counts.
3. State the observed relationship between **Na_to_K and Drug**.
4. State the observed relationship between **BP and Drug**.
5. State whether Age, Sex and Cholesterol appear strongly or weakly associated with the target.
6. State the main numerical correlation observed between Age and Na_to_K.

Do not claim a rule that is not directly supported by the generated plots/counts.

## Task 3 — Missing Values and Outlier Treatment

### Professor requirements
Check missing values with `isnull().sum()`, handle them if present, detect outliers using boxplots/IQR/Z-score, and document the treatment decision.


In [ ]:
# Missing values
missing_values = df.isnull().sum()

print("Missing values per column:")
display(missing_values.to_frame("Missing Values"))

if missing_values.sum() == 0:
    print("No missing values were detected; no imputation was required.")
else:
    print("Missing values are present and will be handled inside the modeling pipeline.")

In [ ]:
# Boxplot-based visual outlier inspection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df["Age"], ax=axes[0])
axes[0].set_title("Outlier Inspection: Age")

sns.boxplot(x=df["Na_to_K"], ax=axes[1])
axes[1].set_title("Outlier Inspection: Na_to_K")

plt.tight_layout()
plt.show()

In [ ]:
# IQR method
def detect_outliers_iqr(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = data[(data[column] < lower) | (data[column] > upper)]

    print(f"--- {column} IQR Analysis ---")
    print(f"Q1: {q1:.3f}")
    print(f"Q3: {q3:.3f}")
    print(f"IQR: {iqr:.3f}")
    print(f"Lower Bound: {lower:.3f}")
    print(f"Upper Bound: {upper:.3f}")
    print(f"Outliers Detected: {len(outliers)}")
    return outliers, lower, upper

age_outliers, age_lower, age_upper = detect_outliers_iqr(df, "Age")
na_k_outliers, na_k_lower, na_k_upper = detect_outliers_iqr(df, "Na_to_K")

print("\nNa_to_K outlier rows:")
display(na_k_outliers[["Age", "Sex", "BP", "Cholesterol", "Na_to_K", "Drug"]])

In [ ]:
# Z-score method
from scipy import stats

z_scores = np.abs(stats.zscore(df["Na_to_K"], nan_policy="omit"))
z_outliers = df[z_scores > 3]

print("Na_to_K Z-score outliers (|Z| > 3):", len(z_outliers))
display(z_outliers[["Age", "Sex", "BP", "Cholesterol", "Na_to_K", "Drug"]])

### Outlier treatment decision

Your previous notebook capped all Na_to_K values using IQR boundaries before the train/test split. That was changed because:

- the IQR bounds were calculated using the full dataset, causing preprocessing leakage;
- a statistical outlier is not automatically a data-entry error;
- the final modeling pipeline should learn preprocessing from the training set only.

Therefore the corrected workflow **keeps the original observations** and performs scaling/transformation inside the training pipeline. The outlier analysis remains documented as required by the assignment.

## Task 4 — Feature Engineering and Preprocessing

### Professor requirements
Encode Sex, BP, Cholesterol and Drug; check skewness; transform when required; scale numerical variables with StandardScaler/MinMaxScaler; split into training/testing datasets.


In [ ]:
# Separate features and target BEFORE any learned transformation.
X = df.drop(columns=["Drug"]).copy()
y = df["Drug"].copy()

print("Feature columns:", X.columns.tolist())
print("Target classes:", sorted(y.unique().tolist()))
print("Target class counts:")
display(y.value_counts().rename("Count").to_frame())

In [ ]:
# Check skewness of numerical features
skewness = X[["Age", "Na_to_K"]].skew()
print("Numerical feature skewness:")
display(skewness.to_frame("Skewness"))

In [ ]:
# Train/test split: 80/20 as stated throughout your existing notebook.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Original dataset size:", len(X))
print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

assert len(X_train) == 160, "Expected 160 training records for an 80/20 split of 200 records."
assert len(X_test) == 40, "Expected 40 testing records for an 80/20 split of 200 records."

In [ ]:
# Standard preprocessing object.
# OneHotEncoder avoids artificial numeric ordering for categorical predictors.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

numeric_features = ["Age", "Na_to_K"]
categorical_features = ["Sex", "BP", "Cholesterol"]

# Na_to_K showed moderate positive skew in your existing analysis.
# We preserve your log1p idea, but apply it ONLY inside the pipeline,
# so the test data is transformed using the same fixed mathematical operation,
# without fitting anything on the test set.
numeric_transformer = Pipeline([
    ("log_na_to_k", FunctionTransformer(
        func=lambda a: np.column_stack([a[:, 0], np.log1p(a[:, 1])]),
        feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

**Important fix to your original code:**  
Do not use one `LabelEncoder` repeatedly on `Sex`, `BP`, `Cholesterol`, and `Drug`. The corrected model keeps the target as the original strings (`DrugA`, `DrugB`, `DrugC`, `DrugX`, `DrugY`) and one-hot encodes only the categorical input variables.

## Task 5 — Data Augmentation (Mandatory)

### Professor requirements
Generate at least 800 synthetic samples so the final dataset reaches at least 1,000; document the augmentation method, number generated and before/after performance.


In [ ]:
# SMOTENC is required here because X contains both numerical and categorical columns.
from imblearn.over_sampling import SMOTENC

categorical_indices = [X_train.columns.get_loc(c) for c in categorical_features]

# Exactly 200 samples per class -> 1,000 total training records.
target_per_class = 200
sampling_strategy = {cls: target_per_class for cls in sorted(y_train.unique())}

print("Original training class distribution:")
display(y_train.value_counts().sort_index().to_frame("Count"))

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=5
)

X_train_aug, y_train_aug = smotenc.fit_resample(X_train, y_train)

print("\nAugmented training class distribution:")
display(y_train_aug.value_counts().sort_index().to_frame("Count"))

synthetic_generated = len(X_train_aug) - len(X_train)

print(f"Original training records : {len(X_train)}")
print(f"Augmented training records: {len(X_train_aug)}")
print(f"Synthetic records generated: {synthetic_generated}")
print(f"Requirement ≥ 800 synthetic: {'MET' if synthetic_generated >= 800 else 'NOT MET'}")
print(f"Requirement ≥ 1,000 final training records: {'MET' if len(X_train_aug) >= 1000 else 'NOT MET'}")

assert len(X_train_aug) == 1000
assert synthetic_generated == 840

In [ ]:
# Visual comparison before vs after augmentation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(x=y_train, ax=axes[0], order=sorted(y_train.unique()))
axes[0].set_title("Before SMOTENC")

sns.countplot(x=y_train_aug, ax=axes[1], order=sorted(y_train_aug.unique()))
axes[1].set_title("After SMOTENC — Balanced to 200/Class")

plt.tight_layout()
plt.show()

In [ ]:
# Save the augmented dataset as required.
OUTPUT_DIR = Path("/content/drive/MyDrive/ML Project")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path(".")

augmented_path = OUTPUT_DIR / "drug_augmented_1000.csv"

df_augmented = X_train_aug.copy()
df_augmented["Drug"] = y_train_aug

df_augmented.to_csv(augmented_path, index=False)

print("Augmented dataset saved to:", augmented_path)
display(df_augmented.head())

### Correct Task 5 conclusion

The corrected split contains 160 original training records and 40 untouched test records. SMOTENC expands the training set to exactly 1,000 records, creating exactly **840 synthetic training records**. The 40-record test set remains untouched and is used for final evaluation. This satisfies the professor's ≥800 synthetic and ≥1,000 total requirements without contaminating the test set.

## Task 6 — Model Building: Try Multiple Classifiers

### Professor requirements
Train Logistic Regression, Decision Tree, Random Forest, SVM, KNN, Naive Bayes, AdaBoost, Gradient Boosting, XGBoost, Extra Trees, Voting and Stacking classifiers.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost is unavailable:", exc)

def make_pipeline(model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

base_lr = LogisticRegression(max_iter=2000, random_state=42)
base_dt = DecisionTreeClassifier(random_state=42, max_depth=None)
base_rf = RandomForestClassifier(n_estimators=200, random_state=42)
base_svm = SVC(probability=True, random_state=42)
base_knn = KNeighborsClassifier()
base_nb = GaussianNB()
base_ada = AdaBoostClassifier(random_state=42)
base_gb = GradientBoostingClassifier(random_state=42)
base_et = ExtraTreesClassifier(n_estimators=200, random_state=42)

model_factories = {
    "Logistic Regression": lambda: make_pipeline(base_lr),
    "Decision Tree": lambda: make_pipeline(base_dt),
    "Random Forest": lambda: make_pipeline(base_rf),
    "SVM": lambda: make_pipeline(base_svm),
    "KNN": lambda: make_pipeline(base_knn),
    "Naive Bayes": lambda: make_pipeline(base_nb),
    "AdaBoost": lambda: make_pipeline(base_ada),
    "Gradient Boosting": lambda: make_pipeline(base_gb),
    "Extra Trees": lambda: make_pipeline(base_et),
}

if XGBOOST_AVAILABLE:
    model_factories["XGBoost"] = lambda: make_pipeline(
        XGBClassifier(
            n_estimators=200,
            random_state=42,
            eval_metric="mlogloss"
        )
    )

# Voting and stacking use their own pipelines as estimators.
voting_estimators = [
    ("lr", make_pipeline(LogisticRegression(max_iter=2000, random_state=42))),
    ("rf", make_pipeline(RandomForestClassifier(n_estimators=200, random_state=42))),
    ("dt", make_pipeline(DecisionTreeClassifier(random_state=42)))
]

stacking_estimators = [
    ("lr", make_pipeline(LogisticRegression(max_iter=2000, random_state=42))),
    ("rf", make_pipeline(RandomForestClassifier(n_estimators=200, random_state=42))),
    ("dt", make_pipeline(DecisionTreeClassifier(random_state=42)))
]

model_factories["Voting Classifier"] = lambda: VotingClassifier(
    estimators=voting_estimators,
    voting="soft"
)

model_factories["Stacking Classifier"] = lambda: StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=42)
)

print("Models included:")
for name in model_factories:
    print(" -", name)

assert len(model_factories) >= 11, "At least 11 required classifier entries should be present."

## Task 6 — Before/After Augmentation Comparison

In [ ]:
# Baseline comparison using Random Forest, as in your existing project.
baseline_model = make_pipeline(
    RandomForestClassifier(n_estimators=200, random_state=42)
)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

baseline_accuracy = accuracy_score(y_test, baseline_pred)

augmented_baseline_model = make_pipeline(
    RandomForestClassifier(n_estimators=200, random_state=42)
)
augmented_baseline_model.fit(X_train_aug, y_train_aug)
augmented_baseline_pred = augmented_baseline_model.predict(X_test)
augmented_accuracy = accuracy_score(y_test, augmented_baseline_pred)

augmentation_improvement = augmented_accuracy - baseline_accuracy

print("========== BEFORE vs AFTER AUGMENTATION ==========")
print(f"Before augmentation accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"After augmentation accuracy : {augmented_accuracy:.4f} ({augmented_accuracy*100:.2f}%)")
print(f"Accuracy change              : {augmentation_improvement:+.4f} ({augmentation_improvement*100:+.2f}%)")
print("==================================================")

## Task 6–9 — One Master Evaluation Engine

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

def evaluate_model(model, X_fit, y_fit, X_eval, y_eval):
    model.fit(X_fit, y_fit)

    train_pred = model.predict(X_fit)
    test_pred = model.predict(X_eval)

    train_acc = accuracy_score(y_fit, train_pred)
    test_acc = accuracy_score(y_eval, test_pred)

    precision = precision_score(y_eval, test_pred, average="weighted", zero_division=0)
    recall = recall_score(y_eval, test_pred, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_eval, test_pred, average="weighted", zero_division=0)
    f1_macro = f1_score(y_eval, test_pred, average="macro", zero_division=0)

    roc_auc = np.nan
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_eval)
            classes = getattr(model, "classes_", np.unique(y_fit))
            roc_auc = roc_auc_score(
                y_eval,
                proba,
                multi_class="ovr",
                average="weighted",
                labels=classes
            )
        except Exception:
            pass

    gap = train_acc - test_acc

    if train_acc < 0.75 and test_acc < 0.75:
        diagnosis = "Underfitting"
    elif gap > 0.10:
        diagnosis = "Overfitting"
    else:
        diagnosis = "No severe overfitting"

    return {
        "model": model,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Weighted": f1_weighted,
        "F1 Macro": f1_macro,
        "ROC-AUC OvR": roc_auc,
        "Gap": gap,
        "Overfitting": diagnosis,
        "Test Predictions": test_pred,
    }

results = []

for name, factory in model_factories.items():
    print(f"Training: {name}")
    try:
        fitted = factory()
        result = evaluate_model(
            fitted,
            X_train_aug,
            y_train_aug,
            X_test,
            y_test
        )
        result["Model"] = name
        results.append(result)
        print(f"  Test Accuracy: {result['Test Accuracy']*100:.2f}%")
    except Exception as exc:
        print(f"  FAILED: {exc}")

assert results, "No models were successfully evaluated."

In [ ]:
# Final model comparison table — NO duplicate Best Model row.
comparison_df = pd.DataFrame([
    {
        "Model": r["Model"],
        "Train Accuracy": r["Train Accuracy"],
        "Test Accuracy": r["Test Accuracy"],
        "Precision": r["Precision"],
        "Recall": r["Recall"],
        "F1 Weighted": r["F1 Weighted"],
        "F1 Macro": r["F1 Macro"],
        "ROC-AUC OvR": r["ROC-AUC OvR"],
        "Augmentation Used": "SMOTENC",
        "Overfitting": "Y" if r["Overfitting"] == "Overfitting" else "N"
    }
    for r in results
]).sort_values(
    by=["F1 Macro", "Test Accuracy"],
    ascending=False
).reset_index(drop=True)

display(
    comparison_df.style.format({
        "Train Accuracy": "{:.2%}",
        "Test Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Weighted": "{:.2%}",
        "F1 Macro": "{:.2%}",
        "ROC-AUC OvR": "{:.2%}",
    })
)

# Select the first model only after considering macro-F1 and test accuracy.
best_model_name = comparison_df.iloc[0]["Model"]
best_result = next(r for r in results if r["Model"] == best_model_name)
best_model = best_result["model"]

print("\nBEST MODEL:", best_model_name)
print("Selection criterion: strongest combination of Macro-F1 and test accuracy.")

## Task 7 — Evaluation and Overfitting Check

### Professor requirements
Report Accuracy, Precision, Recall, F1, ROC-AUC (One-vs-Rest), Confusion Matrix and Classification Report; compare training/testing results for overfitting, underfitting and good generalization.


In [ ]:
# Detailed evaluation of the selected best model
best_test_pred = best_result["Test Predictions"]

print("Classification Report — Best Model")
print(
    classification_report(
        y_test,
        best_test_pred,
        digits=4,
        zero_division=0
    )
)

print("\nTrain Accuracy:", f"{best_result['Train Accuracy']*100:.2f}%")
print("Test Accuracy :", f"{best_result['Test Accuracy']*100:.2f}%")
print("Train-Test Gap:", f"{best_result['Gap']*100:.2f}%")
print("Diagnosis      :", best_result["Overfitting"])

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best_test_pred, labels=sorted(y.unique()))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=sorted(y.unique()),
    yticklabels=sorted(y.unique())
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted Drug")
plt.ylabel("Actual Drug")
plt.tight_layout()
plt.show()

### Corrected overfitting interpretation

Do not automatically claim that a small train/test gap proves the model is clinically reliable. The correct conclusion is limited to this experiment:

> The selected model does/does not show evidence of severe overfitting based on the train-test performance gap. Because the original dataset contains only 200 observations, the performance should be interpreted as an educational machine-learning result rather than clinical validation.

## Task 8 — Hyperparameter Tuning

### Professor requirements
Use GridSearchCV or RandomizedSearchCV, optimize the best-performing models, and record the best hyperparameters and performance change.


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

# Tune Extra Trees as the principal tree-based candidate.
extra_trees_pipeline = make_pipeline(
    ExtraTreesClassifier(random_state=42)
)

et_param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
}

grid_et = GridSearchCV(
    estimator=extra_trees_pipeline,
    param_grid=et_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_et.fit(X_train_aug, y_train_aug)

et_tuned_pred = grid_et.best_estimator_.predict(X_test)
et_tuned_acc = accuracy_score(y_test, et_tuned_pred)

et_baseline_model = make_pipeline(
    ExtraTreesClassifier(n_estimators=200, random_state=42)
)
et_baseline_model.fit(X_train_aug, y_train_aug)
et_baseline_acc = accuracy_score(
    y_test,
    et_baseline_model.predict(X_test)
)

print("Extra Trees GridSearchCV")
print("Best parameters:", grid_et.best_params_)
print(f"Baseline test accuracy: {et_baseline_acc*100:.2f}%")
print(f"Tuned test accuracy   : {et_tuned_acc*100:.2f}%")
print(f"Improvement           : {(et_tuned_acc-et_baseline_acc)*100:+.2f}%")

In [ ]:
# RandomizedSearchCV — second search method, matching your existing project.
et_param_dist = {
    "model__n_estimators": randint(50, 501),
    "model__max_depth": [None, 10, 20, 30, 50],
    "model__min_samples_split": randint(2, 11),
    "model__min_samples_leaf": randint(1, 5),
}

random_et = RandomizedSearchCV(
    estimator=make_pipeline(ExtraTreesClassifier(random_state=42)),
    param_distributions=et_param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_et.fit(X_train_aug, y_train_aug)

random_et_pred = random_et.best_estimator_.predict(X_test)
random_et_acc = accuracy_score(y_test, random_et_pred)

print("Extra Trees RandomizedSearchCV")
print("Best parameters:", random_et.best_params_)
print(f"Baseline test accuracy: {et_baseline_acc*100:.2f}%")
print(f"Tuned test accuracy   : {random_et_acc*100:.2f}%")
print(f"Improvement           : {(random_et_acc-et_baseline_acc)*100:+.2f}%")

In [ ]:
# Tune SVM as the second major candidate.
svm_pipeline = make_pipeline(
    SVC(probability=True, random_state=42)
)

svm_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto", 0.1, 1],
}

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_svm.fit(X_train_aug, y_train_aug)

svm_base_model = make_pipeline(
    SVC(probability=True, random_state=42)
)
svm_base_model.fit(X_train_aug, y_train_aug)

svm_baseline_acc = accuracy_score(
    y_test,
    svm_base_model.predict(X_test)
)

svm_tuned_pred = grid_svm.best_estimator_.predict(X_test)
svm_tuned_acc = accuracy_score(y_test, svm_tuned_pred)

print("SVM GridSearchCV")
print("Best parameters:", grid_svm.best_params_)
print(f"Baseline test accuracy: {svm_baseline_acc*100:.2f}%")
print(f"Tuned test accuracy   : {svm_tuned_acc*100:.2f}%")
print(f"Improvement           : {(svm_tuned_acc-svm_baseline_acc)*100:+.2f}%")

In [ ]:
# Master tuning log — values are calculated, not hard-coded.
tuning_summary_df = pd.DataFrame([
    {
        "Model": "Extra Trees",
        "Search": "GridSearchCV",
        "Best Parameters": str(grid_et.best_params_),
        "Baseline Accuracy": et_baseline_acc,
        "Tuned Accuracy": et_tuned_acc,
        "Improvement": et_tuned_acc - et_baseline_acc
    },
    {
        "Model": "Extra Trees",
        "Search": "RandomizedSearchCV",
        "Best Parameters": str(random_et.best_params_),
        "Baseline Accuracy": et_baseline_acc,
        "Tuned Accuracy": random_et_acc,
        "Improvement": random_et_acc - et_baseline_acc
    },
    {
        "Model": "SVM",
        "Search": "GridSearchCV",
        "Best Parameters": str(grid_svm.best_params_),
        "Baseline Accuracy": svm_baseline_acc,
        "Tuned Accuracy": svm_tuned_acc,
        "Improvement": svm_tuned_acc - svm_baseline_acc
    }
])

display(
    tuning_summary_df.style.format({
        "Baseline Accuracy": "{:.2%}",
        "Tuned Accuracy": "{:.2%}",
        "Improvement": "{:+.2%}"
    })
)

print(
    "Interpret tuning strictly from measured results. "
    "If improvement <= 0, report that tuning did not improve the held-out test accuracy."
)

## Task 9 — Final Model Comparison Table

### Professor requirements
Provide Model, Train Accuracy, Test Accuracy, Precision, Recall, F1 Score, Augmentation Used and Overfitting (Y/N), then identify the Best Model.


In [ ]:
# Export the final comparison table required by the assignment.
comparison_csv = OUTPUT_DIR / "model_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)

print("Saved final model comparison to:", comparison_csv)
display(comparison_df)

## Task 10 — Feature Importance Analysis

### Professor requirements
Identify factors influencing drug selection; provide feature-importance and permutation-importance plots; SHAP is optional.


In [ ]:
# Built-in feature importance for tree-based Extra Trees model.
# Use the best randomized Extra Trees candidate if it exists; otherwise the GridSearch candidate.
final_et_pipeline = random_et.best_estimator_

# Get transformed feature names
prep = final_et_pipeline.named_steps["preprocessor"]
et_model = final_et_pipeline.named_steps["model"]

X_train_aug_transformed = prep.transform(X_train_aug)
X_test_transformed = prep.transform(X_test)

feature_names_transformed = prep.get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "Feature": feature_names_transformed,
    "Importance": et_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Built-in Feature Importance:")
display(feature_importance_df)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=feature_importance_df.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Extra Trees Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance on the ORIGINAL test columns.
perm = permutation_importance(
    final_et_pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)

permutation_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance Mean": perm.importances_mean,
    "Importance Std": perm.importances_std
}).sort_values("Importance Mean", ascending=False)

print("Permutation Importance:")
display(permutation_df)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=permutation_df,
    x="Importance Mean",
    y="Feature"
)
plt.title("Permutation Importance on Untouched Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# Optional SHAP analysis on the transformed matrix.
# SHAP is optional in the professor's task; a failure here does not invalidate
# the mandatory built-in and permutation importance analyses.
try:
    import shap

    explainer = shap.TreeExplainer(et_model)
    shap_values = explainer.shap_values(X_test_transformed)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_test_transformed,
        feature_names=feature_names_transformed,
        plot_type="bar",
        show=False
    )
    plt.title("SHAP Global Feature Importance")
    plt.tight_layout()
    plt.show()

    print("SHAP analysis completed.")
except Exception as exc:
    print("SHAP analysis skipped:", exc)

### Task 10 interpretation

After running the importance cells, write the actual ranking from the output:

- Most influential feature: ______
- Second most influential feature: ______
- Third most influential feature: ______
- Least influential feature: ______

Use the measured feature-importance output rather than copying the typical variables stated in the assignment.

## Task 10/11 — Save the Complete Model Pipeline

In [ ]:
import joblib

# Use the best-performing tuned Extra Trees pipeline as the final deployable model.
final_model = random_et.best_estimator_

MODEL_PATH = OUTPUT_DIR / "best_drug_prediction_pipeline.pkl"
joblib.dump(final_model, MODEL_PATH)

print("Saved COMPLETE preprocessing + model pipeline to:")
print(MODEL_PATH)

## Task 11 — Streamlit Prediction UI

### Professor requirements
Provide Age, Sex, Blood Pressure, Cholesterol and Na_to_K inputs; predict DrugA/DrugB/DrugC/DrugX/DrugY; display predicted drug, prediction probability and confidence percentage.


The Streamlit application is intentionally separated into `app.py`.

**Critical fix compared with your existing app:**  
The saved object is a complete pipeline. Therefore `app.py` accepts the same raw fields used by the original dataset and does not manually recreate LabelEncoder mappings, scaling, or log transformation.

In [ ]:
# Quick deployment smoke test from the notebook.
test_pipeline = joblib.load(MODEL_PATH)

sample_input = pd.DataFrame([{
    "Age": 45,
    "Sex": "Female",
    "BP": "HIGH",
    "Cholesterol": "HIGH",
    "Na_to_K": 15.0
}])

sample_prediction = test_pipeline.predict(sample_input)[0]
sample_probabilities = test_pipeline.predict_proba(sample_input)[0]
sample_classes = test_pipeline.classes_

print("Sample prediction:", sample_prediction)
print("Classes:", sample_classes)
print("Probabilities:", sample_probabilities)
print("Confidence:", f"{sample_probabilities.max()*100:.2f}%")

## Final Validation Checklist

In [ ]:
# Final project validation checks.
print("========== FINAL VALIDATION ==========")
print("Original records:", len(df))
print("Original training records:", len(X_train))
print("Untouched test records:", len(X_test))
print("Augmented training records:", len(X_train_aug))
print("Synthetic records:", synthetic_generated)
print("Best model:", best_model_name)
print("Model file exists:", MODEL_PATH.exists())
print("Augmented CSV exists:", augmented_path.exists())
print("Comparison CSV exists:", comparison_csv.exists())

assert len(df) == 200
assert len(X_train) == 160
assert len(X_test) == 40
assert len(X_train_aug) == 1000
assert synthetic_generated == 840
assert MODEL_PATH.exists()
assert augmented_path.exists()
assert comparison_csv.exists()

print("====================================")
print("ALL CORE PROJECT CHECKS PASSED.")

## Expected Deliverables — Final Check

The professor expects:
- Jupyter Notebook containing complete implementation
- Augmented dataset with at least 1,000 records
- Trained machine-learning model
- Model comparison report
- Streamlit web application for drug prediction
- GitHub repository containing source code and documentation
